# 06 - Text Analytics and Lexicon Sentiment
## Objective 7

**Ahsanullah University of Science and Technology** - Department of Computer Science and Engineering

**Course:** CSE 4262 Data Analytics Lab | **Lab Group:** Gr-03 | **Group:** Gr-06

| Student ID | Name |
|---|---|
| 20220104006 | A.S.M. Tahsin Tajware |
| 20220104014 | Abdullah Al Tamim |
| 20220104032 | Eusha Ahmed Mahi |


### Purpose

One clean token stream is built and reused by the frequency table, CountVectorizer and TF-IDF.
HTML is stripped, tokens are restricted to runs of letters, stop-words are removed and tokens of
two characters or fewer are dropped. Skipping that step fills the vocabulary with `br`,
punctuation-glued words and empty strings, and the frequency table stops meaning anything.

The lexicon sentiment pass is the optional extension. The extract carries no ground-truth
sentiment label, so VADER is scored as an agreement rate against the star rating, not as accuracy.

In [ ]:
import os, sys, glob

_candidates = ["/kaggle/working/repo", "/kaggle/working", "..", "."] + [
    os.path.dirname(p) for p in glob.glob("/kaggle/input/**/da_common.py", recursive=True)]
for _p in _candidates:
    if os.path.exists(os.path.join(_p, "da_common.py")):
        sys.path.insert(0, os.path.abspath(_p))
        break
else:
    raise FileNotFoundError("da_common.py not found. See KAGGLE_SETUP.md for the two setup options.")

from da_common import *

banner("Notebook 06 - Objective 7")
spark = get_spark("06 text analytics")
df = load_analytical(spark).cache()
n_clean = df.count()
print(f"analytical dataset: {n_clean:,} rows")

### 1. Length against star rating

In [ ]:
len_rating = (df.groupBy("rating")
              .agg(F.round(F.avg("review_length")).cast("int").alias("avg_chars"),
                   F.round(F.avg("review_word_count"), 1).alias("avg_words"),
                   F.round(F.expr("percentile_approx(review_length, 0.5)")).cast("int").alias("median_chars"),
                   F.count("*").alias("n"))
              .orderBy("rating"))
len_rating.show()
save_table(len_rating, "obj7_length_by_rating");

### 2. Clean token stream and corpus vocabulary

In [ ]:
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, HashingTF, IDF

body = df.select("rating", F.regexp_replace(F.lower("text"), r"<[^>]+>", " ").alias("body"))
tokenizer = RegexTokenizer(inputCol="body", outputCol="tok",
                           pattern=r"[a-z]+", gaps=False, toLowercase=False)
DOMAIN_STOP = StopWordsRemover().getStopWords() + [
    "get", "got", "would", "one", "use", "used", "im", "dont", "br", "also",
    "really", "even", "well", "much", "product", "item"]
remover = StopWordsRemover(inputCol="tok", outputCol="stopped", stopWords=DOMAIN_STOP)

tokenized = (remover.transform(tokenizer.transform(body))
             .withColumn("clean_tokens", F.expr("filter(stopped, x -> length(x) > 2)"))
             .select("rating", "clean_tokens")).cache()
tokenized.count()

top_terms = (tokenized.select(F.explode("clean_tokens").alias("word"))
             .groupBy("word").count().orderBy(F.desc("count")).limit(20))
print("Top 20 corpus terms"); top_terms.show(20, truncate=False)
save_table(top_terms, "obj7_top_terms");

In [ ]:
terms_by_rating = (tokenized.filter(F.col("rating").isin([1.0, 5.0]))
                   .select("rating", F.explode("clean_tokens").alias("word"))
                   .groupBy("rating", "word").count())
w_term = Window.partitionBy("rating").orderBy(F.desc("count"))
top_by_rating = (terms_by_rating.withColumn("rk", F.row_number().over(w_term))
                 .filter(F.col("rk") <= 12).orderBy("rating", "rk"))
print("Most frequent terms in 1-star vs 5-star reviews")
top_by_rating.show(30, truncate=False)
save_table(top_by_rating, "obj7_top_terms_by_rating");

### 3. CountVectorizer and TF-IDF

In [ ]:
cv_model = CountVectorizer(inputCol="clean_tokens", outputCol="tf",
                           vocabSize=5000, minDF=5.0).fit(tokenized)
tf_df = cv_model.transform(tokenized)

from pyspark.ml.stat import Summarizer
tf_total = tf_df.select(Summarizer.sum(F.col("tf")).alias("s")).first()["s"].toArray()
vocab = np.array(cv_model.vocabulary)
print(f"Vocabulary at minDF=5: {len(vocab):,} terms")

cv_top = (pd.DataFrame({"term": vocab, "corpus_frequency": tf_total.astype("int64")})
          .sort_values("corpus_frequency", ascending=False).head(20).reset_index(drop=True))
save_table(cv_top, "obj7_countvectorizer_top_terms")
save_table(pd.DataFrame([{"vocab_size": int(len(vocab)), "min_df": 5}]), "obj7_vocab_summary")
cv_top

In [ ]:
hashing_tf = HashingTF(inputCol="clean_tokens", outputCol="raw_features", numFeatures=8192)
featurized = hashing_tf.transform(tokenized)
tfidf = IDF(inputCol="raw_features", outputCol="features").fit(featurized).transform(featurized)
print("TF-IDF vectors built on the same cleaned token stream")
tfidf.select("rating", "features").show(3, truncate=70)

In [ ]:
lr, tt = len_rating.toPandas(), top_terms.toPandas()
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].plot(lr["rating"].astype(int), lr["avg_chars"], marker="o", lw=2, color="#db2777", label="Mean")
ax[0].plot(lr["rating"].astype(int), lr["median_chars"], marker="s", lw=2, color="#0891b2",
           ls="--", label="Median")
ax[0].set_title("Review length against star rating")
ax[0].set_xlabel("Stars"); ax[0].set_ylabel("Length (characters)")
ax[0].set_xticks([1, 2, 3, 4, 5]); ax[0].legend()

ax[1].barh(tt["word"][::-1], tt["count"][::-1], color=ACCENT)
ax[1].set_title("Top 20 corpus terms"); ax[1].set_xlabel("Frequency")
ax[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))

plt.tight_layout()
savefig(fig, "fig09_text_analytics", "Objective 7 - length by rating and vocabulary")
plt.show()

### 4. Lexicon sentiment extension

In [ ]:
SAMPLE_FRAC = 0.04
sentiment_done = False
try:
    try:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    except ImportError:
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vaderSentiment"], check=True)
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

    sample = df.select("rating", "text").filter(F.length("text") > 0) \
               .sample(False, SAMPLE_FRAC, seed=7).cache()
    n_s = sample.count()
    analyzer = SentimentIntensityAnalyzer()

    def vader_label(t):
        if not t:
            return "neutral"
        c = analyzer.polarity_scores(t)["compound"]
        return "positive" if c >= 0.05 else ("negative" if c <= -0.05 else "neutral")

    scored = (sample.withColumn("vader", F.udf(vader_label, StringType())(F.col("text")))
                    .withColumn("stars", F.when(F.col("rating") >= 4, "positive")
                                          .when(F.col("rating") <= 2, "negative")
                                          .otherwise("neutral"))).cache()

    agreement = scored.filter(F.col("vader") == F.col("stars")).count() / n_s
    print(f"VADER vs star-rating agreement on {n_s:,} sampled reviews: {agreement:.1%}")

    conf = (scored.groupBy("stars").pivot("vader", ["negative", "neutral", "positive"])
            .count().fillna(0).orderBy("stars"))
    conf.show()

    conf_pdf = save_table(conf, "obj7_vader_confusion")
    save_table(pd.DataFrame([{"sampled_reviews": n_s, "agreement_rate": round(agreement, 4)}]),
               "obj7_vader_agreement")
    sentiment_done = True
except Exception as e:
    print("VADER extension skipped:", type(e).__name__, e)

In [ ]:
if sentiment_done:
    m = conf_pdf.set_index("stars")[["negative", "neutral", "positive"]].astype(float)
    m = m.reindex([r for r in ["negative", "neutral", "positive"] if r in m.index])
    nm = m.div(m.sum(axis=1), axis=0) * 100

    fig, ax = plt.subplots(figsize=(6.2, 4.8))
    im = ax.imshow(nm.values, cmap="Blues", vmin=0, vmax=100)
    ax.set_xticks(range(len(nm.columns))); ax.set_xticklabels(nm.columns)
    ax.set_yticks(range(len(nm.index)));   ax.set_yticklabels(nm.index)
    ax.set_xlabel("VADER label"); ax.set_ylabel("Star-derived label")
    ax.set_title("VADER vs star rating (row-normalised %)"); ax.grid(False)
    for i in range(nm.shape[0]):
        for j in range(nm.shape[1]):
            v = nm.values[i, j]
            ax.text(j, i, f"{v:.0f}%", ha="center", va="center",
                    color="white" if v > 55 else "#1e293b", fontsize=10)
    fig.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    savefig(fig, "fig10_vader_agreement", "Objective 7 - lexicon sentiment vs star rating")
    plt.show()
    sample.unpersist(); scored.unpersist()
else:
    print("Sentiment figure skipped because the VADER pass did not run.")

### Findings

The length pattern is the clearest text result: five-star reviews are markedly shorter than every
other rating. A satisfied customer confirms and leaves; a dissatisfied one explains.

The 1-star versus 5-star term tables are worth putting in the report side by side, since the
vocabulary separates far more cleanly than the overall frequency table suggests.

Read the sentiment agreement rate with care. The star-derived labels are heavily imbalanced toward
positive, so a lexicon that leans positive scores well without being right for the right reason.
The row-normalised confusion matrix shows where it actually fails, which the headline rate hides.


In [ ]:
print(f"Tables in {TBL_DIR}")
for name in sorted(RESULTS):
    print(f"  {name:<40} {len(RESULTS[name]):>6,} rows")
print(f"\nFigures in {FIG_DIR}")
for name in sorted(FIGURES):
    print(f"  {name}.png")

In [ ]:
spark.stop()
print("Spark session stopped.")